In [0]:
%run ./helper

In [0]:
dbutils.widgets.dropdown("source_catalog", catalog_list[0], catalog_list)
dbutils.widgets.dropdown("source_schema", schema_list[0], schema_list)
dbutils.widgets.text("input_table_name", input_table_list[0])
dbutils.widgets.dropdown("write_mode", write_mode_list[0], write_mode_list)
dbutils.widgets.dropdown("max_number_of_topics", "12", ["10", "12", "20"])

catalog=dbutils.widgets.get("source_catalog")
schema=dbutils.widgets.get("source_schema")
input_table_name=dbutils.widgets.get("input_table_name")
write_mode=dbutils.widgets.get("write_mode")
max_number_of_topics=dbutils.widgets.get("max_number_of_topics")
queue_table_name=f"{catalog}.{schema}.{refinement_queue_table_name}"


In [0]:
if not spark.catalog.tableExists(queue_table_name):
    dbutils.notebook.exit(f"Queue table does not exist: {queue_table_name}")


In [0]:
from pyspark.sql.functions import col

processed_count = 0

while True:
    pending_rows = (
        spark.table(queue_table_name)
        .filter((col("status") == "pending") & (col("layer") == 0))
        .orderBy(col("created_at").asc(), col("execution_id").asc(), col("model").asc(), col("topic").asc())
        .limit(1)
        .collect()
    )

    if not pending_rows:
        break

    row = pending_rows[0]
    print(f"Processing execution_id={row['execution_id']}, model={row['model']}, topic={row['topic']}")

    spark.sql(f"""
        UPDATE {queue_table_name}
        SET status = 'running',
            started_at = current_timestamp(),
            finished_at = NULL,
            error_message = NULL
        WHERE execution_id = '{row['execution_id']}'
          AND model = '{row['model']}'
          AND layer = {row['layer']}
          AND topic = '{row['topic']}'
          AND status = 'pending'
    """)

    try:
        result = dbutils.notebook.run(
            "./bertopic",
            0,
            {
                "source_catalog": catalog,
                "source_schema": schema,
                "input_table_name": input_table_name,
                "write_mode": write_mode,
                "cluster_model": row["model"],
                "layer": str(refinement_max_layer),
                "max_number_of_topics": max_number_of_topics,
                "requested_execution_id": row["execution_id"],
                "requested_father_topic": row["topic"]
            }
        )
        processed_count += 1
        print(result)
        spark.sql(f"""
            UPDATE {queue_table_name}
            SET status = 'done',
                finished_at = current_timestamp(),
                error_message = NULL
            WHERE execution_id = '{row['execution_id']}'
              AND model = '{row['model']}'
              AND layer = {row['layer']}
              AND topic = '{row['topic']}'
        """)
    except Exception as e:
        error_message = str(e).replace("'", "''")[:4000]
        spark.sql(f"""
            UPDATE {queue_table_name}
            SET status = 'failed',
                finished_at = current_timestamp(),
                error_message = '{error_message}'
            WHERE execution_id = '{row['execution_id']}'
              AND model = '{row['model']}'
              AND layer = {row['layer']}
              AND topic = '{row['topic']}'
        """)
        raise

display(spark.table(queue_table_name).orderBy(col("created_at").asc(), col("execution_id").asc(), col("topic").asc()))
dbutils.notebook.exit(f"Processed {processed_count} queued refinements")
